# EcoHome Energy Advisor - Agent Run & Evaluation

In this notebook, you'll run the Energy Advisor agent with various real-world scenarios and see how it helps customers optimize their energy usage.

## Learning Objectives
- Create the agent's instructions
- Run the Energy Advisor with different types of questions
- Evaluate response quality and accuracy
- Measure tool usage effectiveness
- Identify areas for improvement
- Implement evaluation metrics

## Evaluation Criteria
- **Accuracy**: Correct information and calculations
- **Relevance**: Responses address the user's question
- **Completeness**: Comprehensive answers with actionable advice
- **Tool Usage**: Appropriate use of available tools
- **Reasoning**: Clear explanation of recommendations


## 1. Import and Initialize

In [1]:
from datetime import datetime
from agent import Agent

In [2]:
ECOHOME_SYSTEM_PROMPT = """You are EcoHome, an AI energy advisor for smart home owners. \
Your expertise covers solar generation, EV charging, HVAC, smart appliances, energy storage, and time-of-use electricity pricing.

Your goal is to help users minimize electricity costs, maximize solar self-consumption, and reduce energy waste \
through specific, data-driven recommendations.

Tools available to you:
- get_weather_forecast: hourly temperature, solar irradiance, and weather conditions by location
- get_electricity_prices: time-of-use (TOU) hourly rates with peak/off-peak/mid-peak windows
- query_energy_usage: historical device consumption by date range and device type
- query_solar_generation: historical solar production records
- get_recent_energy_summary: last N hours of usage and generation in one call
- search_energy_tips: retrieve efficiency tips and best practices from the knowledge base
- calculate_energy_savings: compute exact kWh and USD savings from any optimization scenario

Guidelines:
1. Always call get_weather_forecast and get_electricity_prices before making any scheduling recommendation.
2. Query historical usage or solar data when the question involves patterns, trends, or comparisons.
3. Give concrete answers: exact hours, specific temperatures (°F and °C), dollar amounts, and percentages.
4. Lead every scheduling recommendation with the optimal time window and explain why.
5. Include cost or savings figures in every recommendation — users need numbers to act.
6. Use the location from context when provided; default to typical U.S. TOU pricing when not.
7. When multiple devices compete for off-peak or solar windows, prioritize by energy intensity (kWh/cycle).
8. Keep responses concise and structured. Use short bullet points for lists of recommendations.
"""

In [3]:
ecohome_agent = Agent(
    instructions=ECOHOME_SYSTEM_PROMPT,
)

In [5]:
response = ecohome_agent.invoke(
    question="When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
    context="Location: San Francisco, CA")

In [6]:
print(response["messages"][-1].content)

To minimize cost and maximize solar power when charging your electric car tomorrow, follow these recommendations:

- **Optimal Time Window for Charging: Off-Peak Hours**
  - Off-Peak Hours: 12:00 AM to 6:00 AM
  - Off-Peak Rate: $0.08 per kWh

- **Solar Power Generation Window:**
  - Peak Solar Hours: 8:00 AM to 6:00 PM
  - Highest Solar Irradiance: 813.6 W/m² at 12:00 PM

- **Electricity Pricing Details for Tomorrow:**
  - Peak Hours (Highest Rates): 4:00 PM to 8:00 PM
  - Off-Peak Hours (Lowest Rates): 12:00 AM to 6:00 AM

By charging your electric car during the off-peak hours, you can:
- Save on electricity costs due to the lowest rates during that time.
- Maximize the use of solar power during the day when solar irradiance is high.

Feel free to charge your electric car during the recommended off-peak hours to benefit from lower costs and solar power utilization.


In [7]:

print("TOOLS:")
for msg in response["messages"]:
    if hasattr(msg, "tool_call_id") and msg.tool_call_id:
        print("-", msg.name)


TOOLS:
- get_weather_forecast
- get_electricity_prices


## 2. Define Test Cases

In [8]:
test_cases = [
    {
        "id": "ev_charging_1",
        "question": "When should I charge my electric car tomorrow to minimize cost and maximize solar power?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Should recommend specific hours, mention off-peak rates, and reference solar generation window",
    },
    {
        "id": "thermostat_1",
        "question": "What temperature should I set my thermostat to on Wednesday afternoon when electricity prices spike?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "Should recommend pre-cooling before peak window, give specific setpoint in °F, and mention peak hours",
    },
    {
        "id": "usage_history_1",
        "question": "Suggest three ways I can reduce my energy use based on my usage history.",
        "expected_tools": ["get_recent_energy_summary", "search_energy_tips"],
        "expected_response": "Should list 3 specific recommendations based on device breakdown, with estimated savings per action",
    },
    {
        "id": "dishwasher_savings_1",
        "question": "How much can I save by running my dishwasher during off-peak hours instead of peak hours?",
        "expected_tools": ["get_electricity_prices", "calculate_energy_savings"],
        "expected_response": "Should compute savings in USD using peak vs off-peak rate difference and typical dishwasher kWh usage",
    },
    {
        "id": "pool_pump_1",
        "question": "What is the best time to run my pool pump this week based on the weather forecast?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices"],
        "expected_response": "Should give specific daily time windows combining off-peak rates and solar availability across 3+ days",
    },
    {
        "id": "solar_forecast_1",
        "question": "How much solar power can I expect to generate over the next 3 days?",
        "expected_tools": ["get_weather_forecast", "query_solar_generation"],
        "expected_response": "Should give a daily kWh estimate per day based on irradiance and compare to historical average",
    },
    {
        "id": "hvac_heatwave_1",
        "question": "How should I manage my HVAC system during a heat wave to keep costs down?",
        "expected_tools": ["get_weather_forecast", "get_electricity_prices", "search_energy_tips"],
        "expected_response": "Should recommend pre-cooling schedule, specific setpoints, and HVAC efficiency tips with cost impact",
    },
    {
        "id": "laundry_scheduling_1",
        "question": "When is the best time to run my washing machine and dryer this weekend?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast"],
        "expected_response": "Should suggest specific hours on Saturday/Sunday using off-peak rates and/or solar window",
    },
    {
        "id": "cost_breakdown_1",
        "question": "Give me a breakdown of my energy costs over the past week and flag any unusually high usage.",
        "expected_tools": ["query_energy_usage", "get_recent_energy_summary"],
        "expected_response": "Should show per-device kWh and cost totals, identify the highest consuming device, and flag anomalies",
    },
    {
        "id": "battery_strategy_1",
        "question": "What is the best strategy for charging and discharging my home battery this week?",
        "expected_tools": ["get_electricity_prices", "get_weather_forecast", "search_energy_tips"],
        "expected_response": "Should recommend charge during off-peak or solar peak and discharge during high-rate window with estimated savings",
    },
]

if len(test_cases) < 10:
    raise ValueError("You MUST have at least 10 test cases")

In [9]:
print(f"Loaded {len(test_cases)} test cases.")

Loaded 10 test cases.


## 3. Run Agent Tests

In [10]:
CONTEXT = "Location: San Francisco, CA"

In [11]:
# Run the agent tests
# For each test case, call the agent and collect the response
# Store results for evaluation

print("=== Running Agent Tests ===")
test_results = []

for i, test_case in enumerate(test_cases):
    print(f"\nTest {i+1}: {test_case['id']}")
    print(f"Question: {test_case['question']}")
    print("-" * 50)
    
    try:
        # Call the agent
        response = ecohome_agent.invoke(
            question=test_case['question'],
            context=CONTEXT
        )
        
        # Store the result
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': response,
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat()
        }
        test_results.append(result)
                
    except Exception as e:
        print(f"Error: {e}")
        result = {
            'test_id': test_case['id'],
            'question': test_case['question'],
            'response': f"Error: {str(e)}",
            'expected_tools': test_case['expected_tools'],
            'expected_response': test_case['expected_response'],
            'timestamp': datetime.now().isoformat(),
            'error': str(e)
        }
        test_results.append(result)

print(f"\nCompleted {len(test_results)} tests")


=== Running Agent Tests ===

Test 1: ev_charging_1
Question: When should I charge my electric car tomorrow to minimize cost and maximize solar power?
--------------------------------------------------

Test 2: thermostat_1
Question: What temperature should I set my thermostat to on Wednesday afternoon when electricity prices spike?
--------------------------------------------------

Test 3: usage_history_1
Question: Suggest three ways I can reduce my energy use based on my usage history.
--------------------------------------------------


C:\Users\U329114\OneDrive - IBERDROLA S.A\Formación\Nanodegrees\Agentic AI Engineer\Building AI Agents with LangGraph\ecohome_solution\tools.py:23: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 0.3.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  return OpenAIEmbeddings(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientStartEvent: captur


Test 4: dishwasher_savings_1
Question: How much can I save by running my dishwasher during off-peak hours instead of peak hours?
--------------------------------------------------

Test 5: pool_pump_1
Question: What is the best time to run my pool pump this week based on the weather forecast?
--------------------------------------------------

Test 6: solar_forecast_1
Question: How much solar power can I expect to generate over the next 3 days?
--------------------------------------------------

Test 7: hvac_heatwave_1
Question: How should I manage my HVAC system during a heat wave to keep costs down?
--------------------------------------------------

Test 8: laundry_scheduling_1
Question: When is the best time to run my washing machine and dryer this weekend?
--------------------------------------------------

Test 9: cost_breakdown_1
Question: Give me a breakdown of my energy costs over the past week and flag any unusually high usage.
-----------------------------------------------

In [12]:
test_results

[{'test_id': 'ev_charging_1',
  'question': 'When should I charge my electric car tomorrow to minimize cost and maximize solar power?',
  'response': {'messages': [SystemMessage(content='You are EcoHome, an AI energy advisor for smart home owners. Your expertise covers solar generation, EV charging, HVAC, smart appliances, energy storage, and time-of-use electricity pricing.\n\nYour goal is to help users minimize electricity costs, maximize solar self-consumption, and reduce energy waste through specific, data-driven recommendations.\n\nTools available to you:\n- get_weather_forecast: hourly temperature, solar irradiance, and weather conditions by location\n- get_electricity_prices: time-of-use (TOU) hourly rates with peak/off-peak/mid-peak windows\n- query_energy_usage: historical device consumption by date range and device type\n- query_solar_generation: historical solar production records\n- get_recent_energy_summary: last N hours of usage and generation in one call\n- search_energy

## 4. Evaluate Responses

In [13]:
import os
import json
import re
import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

True

In [14]:
def evaluate_response(question, final_response, expected_response):
    """Evaluate a single response against expected criteria using an LLM judge."""
    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.0,
        base_url="https://openai.vocareum.com/v1",
        api_key=os.getenv("OPENAI_API_KEY")
    )

    prompt = f"""You are evaluating an AI energy advisor response. Score each dimension from 1 (poor) to 5 (excellent).

Question asked: {question}
Expected criteria: {expected_response}
Actual response: {final_response}

Return ONLY valid JSON with no extra text:
{{
  "accuracy": <1-5>,
  "relevance": <1-5>,
  "completeness": <1-5>,
  "actionability": <1-5>,
  "feedback": "<one concise sentence explaining the main strength or weakness>"
}}"""

    result = llm.invoke([HumanMessage(content=prompt)])

    try:
        scores = json.loads(result.content)
    except json.JSONDecodeError:
        match = re.search(r'\{.*\}', result.content, re.DOTALL)
        scores = json.loads(match.group()) if match else {
            "accuracy": 1, "relevance": 1, "completeness": 1, "actionability": 1,
            "feedback": "Could not parse evaluator output"
        }

    scores["composite_score"] = round(
        (scores["accuracy"] + scores["relevance"] + scores["completeness"] + scores["actionability"]) / 4, 2
    )
    return scores

In [18]:
from langchain_core.messages import ToolMessage

def evaluate_tool_usage(messages, expected_tools):
    """Evaluate whether the expected tools were called in the agent's message history."""
    tools_used = []

    for msg in messages:
        if isinstance(msg, ToolMessage):
            if msg.name not in tools_used:
                tools_used.append(msg.name)

    expected = set(expected_tools)
    used = set(tools_used)
    matched = expected & used

    return {
        "tools_used": tools_used,
        "expected_tools": expected_tools,
        "coverage_pct": round(len(matched) / len(expected) * 100, 1) if expected else 100.0,
        "missing_tools": sorted(expected - used),
        "extra_tools": sorted(used - expected),
    }

In [16]:
def generate_evaluation_report():
    rows = []

    for result in test_results:
        if "error" in result:
            rows.append({
                "test_id": result["test_id"],
                "accuracy": 0, "relevance": 0, "completeness": 0, "actionability": 0,
                "composite_score": 0.0,
                "tool_coverage_pct": 0.0,
                "missing_tools": ", ".join(result["expected_tools"]),
                "feedback": result["error"],
                "passed": False
            })
            continue

        messages = result["response"].get("messages", [])
        final_response = messages[-1].content if messages else ""

        response_scores = evaluate_response(result["question"], final_response, result["expected_response"])
        tool_scores = evaluate_tool_usage(messages, result["expected_tools"])

        passed = response_scores["composite_score"] >= 3.0 and tool_scores["coverage_pct"] >= 50.0

        rows.append({
            "test_id": result["test_id"],
            "accuracy": response_scores["accuracy"],
            "relevance": response_scores["relevance"],
            "completeness": response_scores["completeness"],
            "actionability": response_scores["actionability"],
            "composite_score": response_scores["composite_score"],
            "tool_coverage_pct": tool_scores["coverage_pct"],
            "missing_tools": ", ".join(tool_scores["missing_tools"]) if tool_scores["missing_tools"] else "-",
            "feedback": response_scores["feedback"],
            "passed": passed
        })

    df = pd.DataFrame(rows)

    print("=== Evaluation Report ===\n")
    print(df[["test_id", "composite_score", "tool_coverage_pct", "passed"]].to_string(index=False))
    print(f"\nAvg response quality : {df['composite_score'].mean():.2f} / 5.0")
    print(f"Avg tool coverage    : {df['tool_coverage_pct'].mean():.1f}%")
    print(f"Tests passed         : {df['passed'].sum()} / {len(df)}")

    return df

In [19]:
evaluation_df = generate_evaluation_report()

=== Evaluation Report ===

             test_id  composite_score  tool_coverage_pct  passed
       ev_charging_1             4.50              100.0    True
        thermostat_1             4.00               66.7    True
     usage_history_1             3.50              100.0    True
dishwasher_savings_1             4.00               50.0    True
         pool_pump_1             3.25               50.0    True
    solar_forecast_1             2.75               50.0   False
     hvac_heatwave_1             4.00               33.3   False
laundry_scheduling_1             4.50              100.0    True
    cost_breakdown_1             0.00                0.0   False
  battery_strategy_1             4.50               66.7    True

Avg response quality : 3.50 / 5.0
Avg tool coverage    : 61.7%
Tests passed         : 7 / 10
